In [2]:
# Importing env and libaries
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np

In [3]:
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
import os
from pathlib import Path

def load_environment():
    """Load environment variables and return them as a dictionary."""
    load_dotenv(Path("../utils/.env"))  # Loads .env from project root (works if run from notebook too)
    env_vars = {
        "IMAGE_FOLDER": os.getenv("IMAGE_FOLDER"),
        "EXTRACTEDDATASET_FOLDER": os.getenv("EXTRACTEDDATASET_FOLDER"),
        "DATASETS_FOLDER": os.getenv("DATASETS_FOLDER"),
        "ElevationDataset": os.getenv("ElevationDataset"),
        "LandCoverDataset": os.getenv("LandCoverDataset"),
        "GeoBoundaries": os.getenv("GeoBoundaries"),
        "EXTRACTEDELEVATION_FOLDER": os.getenv("EXTRACTEDELEVATION_FOLDER"),
        "EXTRACTEDLANDCOVER_FOLDER": os.getenv("EXTRACTEDLANDCOVER_FOLDER"),
        "EXTRACTEDGEOBOUNDARIES_FOLDER": os.getenv("EXTRACTEDGEOBOUNDARIES_FOLDER"),
        "CLEANEDDATASET_FOLDER": os.getenv("CLEANEDDATASET_FOLDER"),
        "CLEANEDELEVATION_FOLDER": os.getenv("CLEANEDELEVATION_FOLDER"),
        "CLEANEDLANDCOVER_FOLDER": os.getenv("CLEANEDLANDCOVER_FOLDER"),
        "PREPROCESSED_DATASET_FOLDER": os.getenv("PREPROCESSED_DATASET_FOLDER"),
        "PREPROCESSED_ELEVATION_FOLDER": os.getenv("PREPROCESSED_ELEVATION_FOLDER"),
        "PREPROCESSED_LANDCOVER_FOLDER": os.getenv("PREPROCESSED_LANDCOVER_FOLDER")
    }
    return env_vars

folders = load_environment()
image_folder = folders["IMAGE_FOLDER"]
extractedData_folder = folders["EXTRACTEDDATASET_FOLDER"]
datasets_folder = folders["DATASETS_FOLDER"]
landCover_folder = folders["LandCoverDataset"]
elevation_folder = folders["ElevationDataset"]
geoboundaries_folder = folders["GeoBoundaries"]
extracted_elevation_folder = folders["EXTRACTEDELEVATION_FOLDER"]
extracted_landcover_folder = folders["EXTRACTEDLANDCOVER_FOLDER"]
extracted_geo_boundaries_folder = folders["EXTRACTEDGEOBOUNDARIES_FOLDER"]
cleaned_dataset_folder = folders["CLEANEDDATASET_FOLDER"]
cleaned_elevation_folder = folders["CLEANEDELEVATION_FOLDER"]
cleaned_landcover_folder = folders["CLEANEDLANDCOVER_FOLDER"]
preprocessed_dataset_folder = folders["PREPROCESSED_DATASET_FOLDER"]
preprocessed_elevation_folder = folders["PREPROCESSED_ELEVATION_FOLDER"]
preprocessed_landcover_folder = folders["PREPROCESSED_LANDCOVER_FOLDER"]


## performing a join

In [5]:
fire_df_final = gpd.read_file(f"{preprocessed_dataset_folder}/FireDataset/fire_type_1.geojson")

In [6]:
landcover = gpd.read_file(f"{preprocessed_landcover_folder}/preprocessed_landcover")

### filtering to take mostly north ( we will do this later)

In [ ]:
# # filtering in the range
# # Define the coordinates
# min_lat = 26
# max_lat = 37.32346
# min_lon = -2.5
# max_lon = 11.11035

# # GeoPandas uses (minx, miny, maxx, maxy) for the .cx indexer,
# # which corresponds to (min_lon, min_lat, max_lon, max_lat)
# landcover_filtered = landcover.cx[min_lon:max_lon, min_lat:max_lat]


In [ ]:
# (len(landcover_filtered) / len(landcover) ) * 100

64.91118849384169

kept about 65% of landcover data 

In [ ]:
# len(landcover_filtered) 

284644

In [ ]:
len(fire_df_final)

14216

### applying a 0.1 grid

In [10]:
landcover.columns

Index(['area', 'lcc_Bare lands', 'lcc_Croplands', 'lcc_Forests',
       'lcc_Grasslands', 'lcc_Vegetation', 'lcc_Water bodies', 'geometry'],
      dtype='object')

In [12]:
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
import pandas as pd # Needed for the mode calculation and potential encoding

# Assuming landcover is the Land Cover GeoDataFrame loaded earlier.
# IMPORTANT: For this code to run, 'landcover' MUST contain the single categorical column 'lcc_highlevel'.

# --- 1. FISHNET GRID CREATION ---
# 1. Get the bounding box of the entire study area
minx, miny, maxx, maxy = landcover.total_bounds

# 2. Define the cell size
cell_size = 0.1

# 3. Create grid coordinates
x_coords = np.arange(minx, maxx, cell_size)
y_coords = np.arange(miny, maxy, cell_size)

polygons = []

# 4. Create the polygon geometries
for x in x_coords:
    for y in y_coords:
        polygons.append(Polygon([
            (x, y), 
            (x + cell_size, y), 
            (x + cell_size, y + cell_size), 
            (x, y + cell_size)
        ]))

# 5. Create the grid GeoDataFrame
grid_gdf = gpd.GeoDataFrame(
    {'grid_id': range(len(polygons))}, 
    geometry=polygons, 
    crs=landcover.crs
)
# ... (Steps 1 to 5 for grid_gdf creation are correct) ...

# --- 2. SPATIAL JOIN ---

# Select all non-geometry columns plus the geometry column for the join
feature_cols = [col for col in landcover.columns if col != 'geometry'] + ['geometry']

grid_with_landcover = gpd.sjoin(
    grid_gdf,
    landcover[feature_cols],
    how='left',
    # FIX: Replace 'op' with the current GeoPandas keyword 'predicate'
    predicate='intersects' 
)

# --- 3. AGGREGATION ---
# Now we must aggregate ALL the Land Cover features by their mean/median.
# We use 'mean' for both 'area' and the one-hot encoded 'lcc_...' columns.

agg_operations = {
    'geometry': 'first', # Keep the original polygon geometry for the cell
    'area': 'mean'       # Mean of the area of overlapping polygons
}
# Add 'mean' operation for all six one-hot encoded columns
for col in landcover.columns:
    if col.startswith('lcc_'):
        agg_operations[col] = 'mean'

# 2. Aggregate the data
final_landcover_grid = grid_with_landcover.groupby('grid_id').agg(agg_operations).reset_index()

# 3. Convert back to GeoDataFrame
final_landcover_grid = gpd.GeoDataFrame(
    final_landcover_grid, 
    geometry='geometry', 
    crs=landcover.crs 
)

print(f"Final Land Cover Grid created with {len(final_landcover_grid)} cells.")

Final Land Cover Grid created with 38502 cells.


The fire_merged_with_landcover GeoDataFrame now contains all your fire data with the corresponding land cover feature.

In [15]:
# Assuming 'fire_df_final' contains the preprocessed fire points (Point geometry).
# Assuming 'final_landcover_grid' contains the final aggregated/encoded grid polygons (Polygon geometry).

# Perform the Spatial Join
# The predicate='within' ensures the fire point is linked to the grid polygon that contains it.
fire_merged_with_landcover = gpd.sjoin(
    left_df=fire_df_final,
    right_df=final_landcover_grid,
    how='left',
    predicate='within' # <-- CORRECTED: Changed 'op' to 'predicate'
)

# Clean up the extraneous column created by the sjoin operation
fire_merged_with_landcover = fire_merged_with_landcover.drop(columns=['index_right'])

print("Spatial join complete. Fire points merged with Land Cover features.")

Spatial join complete. Fire points merged with Land Cover features.


In [16]:
fire_merged_with_landcover.columns

Index(['fire', 'geometry', 'grid_id', 'area', 'lcc_Bare lands',
       'lcc_Croplands', 'lcc_Forests', 'lcc_Grasslands', 'lcc_Vegetation',
       'lcc_Water bodies'],
      dtype='object')

In [17]:
fire_merged_with_landcover.head()

,fire,geometry,grid_id,area,lcc_Bare lands,lcc_Croplands,lcc_Forests,lcc_Grasslands,lcc_Vegetation,lcc_Water bodies
0,1,POINT (5.53337 35.70751),26579,1.102550e+08,0.0,0.54321,0.0,0.098765,0.358025,0.0
1,1,POINT (6.46961 32.13579),28217,6.174165e+10,1.0,0.0,0.0,0.0,0.0,0.0
2,1,POINT (6.9763 32.35563),29149,1.852243e+11,1.0,0.0,0.0,0.0,0.0,0.0
3,1,POINT (9.39581 28.19791),33572,6.720004e+11,1.0,0.0,0.0,0.0,0.0,0.0
4,1,POINT (9.49323 28.12826),33757,1.120028e+11,1.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# Save the final aggregated/encoded Land Cover grid
final_landcover_grid.to_file("../../MergedDatasets/final_landcover_grid.gpkg", driver="GPKG")
print("Saved final_landcover_grid.gpkg")

Saved final_landcover_grid.gpkg


In [19]:
# Assuming your final merged dataset is named 'fire_merged_with_landcover'
fire_merged_with_landcover.to_file("../../MergedDatasets/fire_features_lc_merged.gpkg", driver="GPKG")
print("Saved fire_features_lc_merged.gpkg")

Saved fire_features_lc_merged.gpkg


In [20]:
print(len(fire_merged_with_landcover))

14216
